# Title: 01_feature_extraction
### Goal: Build _data/processed/features.csv_ from ENCODE bigWig trakcs over safe harbor + matched control windows.

In [16]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import requests
import pyranges as pr
import gzip

PROJECT_ROOT = Path("..").resolve()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT

PosixPath('/home/mazani/projects/safe-harbor-ml')

## Define anchor loci (hg38)

In [26]:
# Known safe harbor anchor regions (hg38)
# These serve as starting points for fixed-size windows

positives = pd.DataFrame(
    [
      # name, chromosome, start end (hg38)
        ("AAVS1_PPP1R12C", "chr19", 55115996, 55115997),
        ("CCR5", "chr3", 46369996, 46371554),
        ("ROSA26_like_THUMPD3_AS1", "chr3", 9349689, 9398625),
    ],
    columns=["locus", "Chromosome", "Start", "End"]
    )

positives

,locus,Chromosome,Start,End
0,AAVS1_PPP1R12C,chr19,55115996,55115997
1,CCR5,chr3,46369996,46371554
2,ROSA26_like_THUMPD3_AS1,chr3,9349689,9398625


## Convert anchor loci into fixed-size genomic windows

In [27]:
WINDOW_BP = 2000 # 2 kb window

def anchors_to_windows(df: pd.DataFrame, window_bp: int) -> pd.DataFrame:
    """
    Convert anchor regions into fixed-size windows centered on the anchor midpoint.
    """

    out = df.copy()

    midpoints = ((out["Start"] + out["End"]) // 2).astype(int)
    half = window_bp // 2

    out["Start"] = (midpoints - half).clip(lower=0)
    out["End"] = midpoints + half

    return out

pos_windows = anchors_to_windows(positives, WINDOW_BP)
pos_windows["label"] = 1
pos_windows["tier"] = "T1"

pos_windows

,locus,Chromosome,Start,End,label,tier
0,AAVS1_PPP1R12C,chr19,55114996,55116996,1,T1
1,CCR5,chr3,46369775,46371775,1,T1
2,ROSA26_like_THUMPD3_AS1,chr3,9373157,9375157,1,T1


## Downlaod/load ENCODE hg38 blacklist

In [28]:
blacklist_path = DATA_RAW / "hg38-blacklist.bed"

if not blacklist_path.exists():
    url = "https://www.encodeproject.org/files/ENCFF356LFX/@@download/ENCFF356LFX.bed.gz"
    gz_path = DATA_RAW / "ENCFF356LFX.bed.gz"

    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    with open(gz_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)

    with gzip.open(gz_path, "rt") as fin, open(blacklist_path, "w") as fout:
        for line in fin:
            fout.write(line)

blacklist = pr.read_bed(str(blacklist_path))
blacklist

,Chromosome,Start,End
0,chr1,628903,635104
1,chr1,5850087,5850571
2,chr1,8909610,8910014
3,chr1,9574580,9574997
4,chr1,32043823,32044203
...,...,...,...
905,chrY,11290797,11334278
906,chrY,11493053,11592850
907,chrY,11671014,11671046
908,chrY,11721528,11749472


## Convert positives to PyRanges

In [30]:
pos_gr = pr.PyRanges(pos_windows[["Chromosome", "Start", "End", "locus", "label", "tier"]])
pos_gr

,Chromosome,Start,End,locus,label,tier
0,chr3,46369775,46371775,CCR5,1,T1
1,chr3,9373157,9375157,ROSA26_like_THUMPD3_AS1,1,T1
2,chr19,55114996,55116996,AAVS1_PPP1R12C,1,T1


## Generate Matched Negatives

In [41]:
NEG_PER_POS_FINAL = 300
FLANK_BP = 5_000_000  # sample within +/- 5 Mb of each positive (keeps chrom context somewhat similar)

def make_negative_pool(pos_df, n_per_pos=1000, flank_bp=5_000_000, window_bp=2000):
    half = window_bp // 2
    rows = []

    for _, row in pos_df.iterrows():
        chrom = row["Chromosome"]
        mid = (row["Start"] + row["End"]) // 2

        for _ in range(n_per_pos):
            new_mid = mid + random.randint(-flank_bp, flank_bp)
            new_mid = max(new_mid, half + 1)
            rows.append((chrom, new_mid - half, new_mid + half))

    neg = pd.DataFrame(rows, columns=["Chromosome", "Start", "End"])
    neg["label"] = 0
    neg["tier"] = "NEG"
    neg["locus"] = "NEG"
    return neg

neg_pool = make_negative_pool(pos_windows, n_per_pos=2000, flank_bp=FLANK_BP, window_bp=WINDOW_BP)
neg_gr = pr.PyRanges(neg_pool)

# Remove overlaps with blacklist and positives
neg_gr = neg_gr.subtract(blacklist).subtract(pos_gr)

# subtract can fragment intervals; keep only intervals that still match the intended window length
neg_df = neg_gr.df
neg_df["len"] = neg_df["End"] - neg_df["Start"]
neg_df = neg_df[(neg_df["len"] >= int(WINDOW_BP * 0.95)) & (neg_df["len"] <= int(WINDOW_BP * 1.05))].drop(columns=["len"])

# Sample final negatives
target_neg = len(pos_windows) * NEG_PER_POS_FINAL
neg_final = neg_df.sample(n=min(target_neg, len(neg_df)), random_state=SEED).reset_index(drop=True)

# Combine
windows = pd.concat(
    [
        pos_windows[["Chromosome", "Start", "End", "locus", "label", "tier"]],
        neg_final[["Chromosome", "Start", "End", "locus", "label", "tier"]],
    ],
    ignore_index=True
)

windows.shape, windows["label"].value_counts()

((903, 6),
 label
 0    900
 1      3
 Name: count, dtype: int64)

## Search ENCODE for bigWIG tracks (Cell Line: K562)

In [44]:
ENCODE_HEADERS = {"accept": "application/json"}

def encode_file_search(query: str, limit: int=50) -> dict:
    url = f"https://www.encodeproject.org/search/?type=File&format=json&{query}&limit={limit}"
    r = requests.get(url, headers=ENCODE_HEADERS, timeout=60)
    r.raise_for_status()
    return r.json()

def extract_bigwig_downloads(search_json: dict, assembly="GRCh38") -> pd.DataFrame:
    rows = []
    for g in search_json.get("@graph", []):
        if g.get("file_format") != "bigWig":
            continue
        if g.get("assembly") != assembly:
            continue
        rows.append({
            "accession": g.get("accession"),
            "output_type": g.get("output_type"),
            "assay_title": g.get("assay_title"),
            "target": (g.get("target") or {}).get("label"),
            "href": g.get("href"),
            "status": g.get("status"),
        })
    return pd.DataFrame(rows)

queries = {
    "ATAC": "biosample_ontology.term_name=K562&assay_title=ATAC-seq&file_format=bigWig",
    "H3K27ac": "biosample_ontology.term_name=K562&assay_title=ChIP-seq&target.label=H3K27ac&file_format=bigWig",
    "H3K4me3": "biosample_ontology.term_name=K562&assay_title=ChIP-seq&target.label=H3K4me3&file_format=bigWig",
}

all_tracks = []
for mark, q in queries.items():
    js = encode_file_search(q, limit=50)
    df = extract_bigwig_downloads(js, assembly="GRCh38")
    df["mark"] = mark
    all_tracks.append(df)


tracks_df = pd.concat(all_tracks, ignore_index=True)
tracks_df.head(20)

HTTPError: 404 Client Error: Not Found for url: https://www.encodeproject.org/search/?type=File&format=json&biosample_ontology.term_name=K562&assay_title=ChIP-seq&target.label=H3K27ac&file_format=bigWig&limit=50